# 05 ? Gera??o offline versus on-the-fly: throughput e espera da GPU

Compara a gera??o reproduz?vel dos mesmos batches de seis canais nos modos online e materializado offline. N?o mede qualidade de forecasting; a pergunta ? custo de prepara??o, throughput de treino e per?odos em que a GPU aguarda batches.

## Registro do piloto

- Checkpoint fixado: `amazon/chronos-2` @ `29ec3766d36d6f73f0696f85560a422f50e8498c`; configura??o: contexto m?x. 8192, sa?da patch 16, at? 1024 passos.
- Estado do ambiente atual: CPU-only; o trace CUDA fica pendente de m?quina com CUDA. O notebook 08 ? apenas microbenchmark CPU.

- Status: harness de benchmark. Executar primeiro no ambiente CUDA com checkpoint Chronos-2 fixado.
- Gerador: tsfmxai; mesmo `OnlineBatchPlan`, seeds por batch ID e sequ?ncia de batches nos dois modos.
- Modelo: mesmo Chronos-2/checkpoint, treino sem equation head para isolar gera??o; warm-up n?o entra na m?dia.
- Formato proposto: 8 exemplos multivariados ? 6 canais, contexto 512, horizonte 64.
- Modos online: `DataLoader` com `num_workers` configur?vel, prefetch limitado e workers persistentes.
- Offline: gerar os batches uma vez e armazenar em cache local ignorado; medir prepara??o e treino separadamente.
- GPU stall ser? estimado pelo trace CPU/CUDA: tempo esperando `next(loader)` n?o ser? rotulado diretamente como ociosidade da GPU.


## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/05_offline_vs_onthefly.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.
The Chronos Python implementation is read from `experiments_local/chronos-forecasting`; recreate that ignored checkout at commit `10afa9ebe016e514f9d7dc1aa873f66af57e116b` and install it with `python -m pip install -e experiments_local/chronos-forecasting`.


In [1]:
from dataclasses import dataclass
from pathlib import Path
import time
import numpy as np
import torch

ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/'src/tsfmxai').exists(): raise RuntimeError('repository root not found')
import sys
sys.path.insert(0,str(ROOT/'src'))
from tsfmxai.datasets.online import OnlineBatchPlan, CurriculumStage, make_dataloader
from tsfmxai.datasets.multiseries import TrainingFormat, TrajectoryConfig
from tsfmxai.generators.initialization import HistorySpec
from tsfmxai.generators.combinatorial_expression import CombinatorialTreeConfig
from tsfmxai.generators.multiseries import MultiseriesConfig, NoiseConfig

MODEL_ID='amazon/chronos-2'
MODEL_REVISION='29ec3766d36d6f73f0696f85560a422f50e8498c'
OUTPUT_PATCH_SIZE=16
SEED=20261120
TOTAL_BATCHES=32  # pilot only; increase after hardware is chosen
EXAMPLES_PER_BATCH=8
CHANNELS=6
CONTEXT=512
HORIZON=64
WORKER_COUNTS=(0,1,2,4)
PREFETCH_FACTOR=2
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

stage_config=MultiseriesConfig(
 ast_config=CombinatorialTreeConfig(minimum_operators=1,maximum_operators=3,minimum_lag=1,maximum_lag=16,max_degree=2),
 channel_count_probs={CHANNELS:1.0},dependency_probs={'independent':1.0},
 treatment_probs={'shao_wu':1.0},max_external_sources=0,
 noise=NoiseConfig(std_range=(0.01,0.1)))
plan=OnlineBatchPlan(
 stages=(CurriculumStage(0,stage_config,EXAMPLES_PER_BATCH),),
 total_batches=TOTAL_BATCHES,root_seed=SEED,
 trajectory=TrajectoryConfig(CONTEXT+HORIZON+64,HistorySpec('uniform','uniform')),
 training=TrainingFormat(CONTEXT,HORIZON,64,batch_size=EXAMPLES_PER_BATCH,dtype='float32'))
print({'device':DEVICE,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'rows_per_forward':EXAMPLES_PER_BATCH*CHANNELS,'batches':TOTAL_BATCHES,
       'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'output_patch_size':OUTPUT_PATCH_SIZE,
       'dependency':'independent','treatment':'shao_wu'})

{'device': 'cpu', 'torch': '2.12.0+cpu', 'cuda': False, 'rows_per_forward': 48, 'batches': 32, 'model_id': 'amazon/chronos-2', 'model_revision': '29ec3766d36d6f73f0696f85560a422f50e8498c', 'output_patch_size': 16, 'dependency': 'independent', 'treatment': 'shao_wu'}


## Comparabilidade e corre??o dos exemplos

O gerador deve produzir a mesma sequ?ncia l?gica de batches nos dois modos. Cada batch ID determina a seed, e o armazenamento offline apenas materializa `plan.generate(id)`. Antes de medir, comparar campo a campo um conjunto de IDs: contexto, futuro, skeleton targets e std precisam ser iguais.

O gerador cont?nuo atual emite batches por quantidade de janelas; um batch ID pode corresponder a um n?mero vari?vel de mecanismos se sistemas falharem ou produzirem janelas insuficientes. O piloto abaixo exige shape constante e deve falhar cedo se isso ocorrer. Para treino longo, preferir uma API que defina exatamente o tamanho do batch ou registre/rejeite o ID inteiro, nunca altere a composi??o silenciosamente.

In [2]:
def materialize(plan, directory):
    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)
    start=time.perf_counter()
    for batch_id in range(plan.total_batches):
        batch=plan.generate(batch_id)
        np.savez_compressed(directory/f'batch-{batch_id:06d}.npz',
            context=batch.context,future=batch.future,channel_mask=batch.channel_mask,
            equation_targets=batch.equation_targets,equation_mask=batch.equation_mask,
            noise_std=batch.noise_targets['std'])
    return time.perf_counter()-start

# Keep generated data local/ignored. Call after choosing disk/cache location.
OFFLINE_DIR=ROOT/'outputs'/'generator_model_evaluation'/'pilot_cache'
# offline_preparation_seconds = materialize(plan, OFFLINE_DIR)

# TODO for selected IDs: load NPZ batches and assert exact equality with plan.generate(id).
# For the actual throughput run, use the same model/checkpoint and update count.

def load_cached_batch(directory, batch_id):
    with np.load(Path(directory)/f'batch-{batch_id:06d}.npz') as f:
        return {key: f[key].copy() for key in f.files}

def assert_cache_matches_plan(plan, directory, batch_ids):
    fields=('context','future','channel_mask','equation_targets','equation_mask')
    for batch_id in batch_ids:
        generated=plan.generate(batch_id)
        cached=load_cached_batch(directory,batch_id)
        for key in fields: assert np.array_equal(getattr(generated,key),cached[key]),(batch_id,key)
        assert np.array_equal(generated.noise_targets['std'],cached['noise_std'])
    return True


## Instrumenta??o de espera e profiling

Medir `next(iterator)` com rel?gio monot?nico d? o bloqueio observado pelo processo de treino, mas CUDA ? ass?ncrono. Para atribuir stall, capturar poucos passos com `torch.profiler` e atividades CPU/CUDA ap?s warm-up, inserindo regi?es `data_wait`, `h2d`, `forward`, `backward` e `optimizer`. Inspecionar a linha do tempo: stall ? um intervalo sem kernels em que o pr?ximo batch ainda n?o chegou.

Fazer duas passagens: (A) profiler curto para diagn?stico, (B) execu??o sem profiler para throughput. N?o sincronizar CUDA a cada batch no benchmark principal, porque isso remove sobreposi??o. Registrar mediana/p95 da espera, batches/s, exemplos/s, canais v?lidos/s, prepara??o offline, treino, total e pico de RAM/VRAM.


In [3]:
def time_batch_wait(loader, warmup=4, measured=20):
    iterator=iter(loader)
    for _ in range(warmup): next(iterator)
    waits=[]
    for _ in range(measured):
        t0=time.perf_counter(); batch=next(iterator); waits.append(time.perf_counter()-t0)
    waits=np.asarray(waits)
    return {'wait_median_s':float(np.median(waits)),
            'wait_p95_s':float(np.quantile(waits,.95)),
            'batches_per_s':float(1/waits.mean())}

# CUDA profiler recipe (run on the training loop, not just data iteration):
# activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
# with torch.profiler.profile(activities=activities, record_shapes=True) as prof:
#     with torch.profiler.record_function('data_wait'): batch=next(iterator)
#     with torch.profiler.record_function('h2d'): batch=move_batch(batch, DEVICE)
#     with torch.profiler.record_function('forward_backward_optimizer'): train_step(batch)
# prof.export_chrome_trace(str(ROOT/'outputs/generator_model_evaluation/online-trace.json'))
# Current environment is CPU-only; don't interpret CPU timing as GPU-stall measurement.

## Matriz do benchmark e decis?o

Executar online com workers `0, 1, 2, 4` e prefetch `1, 2`; medir cache offline e treino offline. Repetir ao menos tr?s vezes, alternando a ordem dos modos e limpando/registrando estado de cache. Manter os mesmos batches, updates, seed, modelo e tamanhos. Relatar tempo de prepara??o e treinamento separadamente e tamb?m soma dos dois.

A conclus?o de stall exige profiler com GPU; `data_wait` isolado ? apenas proxy. Se o throughput online se aproximar do offline e a GPU timeline n?o mostrar lacunas relevantes, aumentar workers n?o traz benef?cio pr?tico. Se a gera??o ficar atr?s, perfilar o executor AST e otimizar a simula??o em lote antes de aumentar indefinidamente processos.